# CodeCrafters webinar evaluation: CRISP-DM mixed-methods analysis

This notebook is the execution entry point for the reproducible analysis pipeline. It follows CRISP-DM: business understanding, data understanding, data preparation, descriptive modeling/analysis, evaluation, and deployment. Run it with the local Conda/Jupyter kernel when you need generated outputs to appear in this Git working tree. The raw CSV is read-only; derived tables and figures are written under `ANALYSIS/outputs/`.

## Colab setup

The preferred execution mode is the local Conda/Jupyter kernel because it shares this VS Code workspace and writes `ANALYSIS/outputs/` directly into the repository. A Colab kernel remains supported as a remote fallback, but its files must be exported back before they can be committed.

In [ ]:
import importlib.util

required_packages = ['pandas', 'numpy', 'matplotlib', 'seaborn']
missing_packages = [name for name in required_packages if importlib.util.find_spec(name) is None]
if missing_packages:
    raise ImportError(f'Missing packages in the active Conda/Jupyter environment: {missing_packages}. Install them in the active environment only if necessary.')
print('Required analysis packages are available in the active kernel; no notebook installation was run.')

In [ ]:
from pathlib import Path
import sys
import subprocess
import zipfile

def is_project_root(candidate):
    analysis_dir = candidate / 'ANALYSIS'
    return (analysis_dir / 'src').is_dir() and (
        (analysis_dir / 'data').is_dir() or (candidate / 'data').is_dir()
    )

def find_project_root(search_roots):
    for search_root in search_roots:
        if not search_root.exists():
            continue
        upward_candidates = [search_root, *search_root.parents]
        for candidate in upward_candidates:
            if is_project_root(candidate):
                return candidate
        for analysis_dir in search_root.rglob('ANALYSIS'):
            candidate = analysis_dir.parent
            if is_project_root(candidate):
                return candidate
    return None

# Local Conda/Jupyter is the default. This path is optional when the notebook
# is opened from inside the repository; the upward search normally finds it.
LOCAL_PROJECT_ROOT = None

# The Git/ZIP settings are used only when the active kernel is genuine Colab.
REPOSITORY_URL = 'https://github.com/adriaanamd/operation-brea-olla-iv-a.git'
REPOSITORY_REF = 'master'
REMOTE_CLONE_DIR = Path('/content/organized-webminar-manuscript')

if LOCAL_PROJECT_ROOT is not None:
    PROJECT_ROOT = Path(LOCAL_PROJECT_ROOT).expanduser().resolve()
else:
    PROJECT_ROOT = find_project_root([Path.cwd()])

is_colab_runtime = False
try:
    import google.colab  # type: ignore
    is_colab_runtime = True
except ImportError:
    pass

if PROJECT_ROOT is None and is_colab_runtime:
    # Search the remote Colab filesystem only after local discovery fails.
    PROJECT_ROOT = find_project_root([Path('/content')])
    if PROJECT_ROOT is None:
        clone_command = [
            'git', 'clone', '--depth', '1', '--branch', REPOSITORY_REF,
            REPOSITORY_URL, str(REMOTE_CLONE_DIR),
        ]
        clone_result = subprocess.run(clone_command, capture_output=True, text=True)
        if clone_result.returncode == 0:
            PROJECT_ROOT = REMOTE_CLONE_DIR
        else:
            print('Git clone failed; upload a ZIP of the repository instead.')

if PROJECT_ROOT is None and is_colab_runtime:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            'The Colab kernel is remote, so it cannot see the local VS Code workspace. '
            'Use the local Conda/Jupyter kernel for repository-local outputs, or upload a ZIP. '
            'Do not place a GitHub token or password in this notebook.'
        ) from exc
    print('No manuscript repository found. Upload a ZIP containing ANALYSIS/src and data.')
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if not zip_names:
        raise FileNotFoundError('Upload a ZIP of the manuscript repository, not only the notebook file.')
    for zip_name in zip_names:
        with zipfile.ZipFile(zip_name) as archive:
            archive.extractall('/content')
    PROJECT_ROOT = find_project_root([Path('/content')])

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'The active kernel cannot locate the manuscript repository. In local Conda/Jupyter, '
        'open the notebook from the repository or set LOCAL_PROJECT_ROOT to the local manuscript folder.'
    )

sys.path.insert(0, str(PROJECT_ROOT))
data_dir = PROJECT_ROOT / 'ANALYSIS' / 'data'
if not data_dir.is_dir():
    data_dir = PROJECT_ROOT / 'data'
data_files = sorted(data_dir.glob('*.csv'))
if len(data_files) != 1:
    raise FileNotFoundError(f'Expected exactly one CSV in ANALYSIS/data; found {len(data_files)}.')
input_path = data_files[0]
print('Project root:', PROJECT_ROOT)
print('Data directory:', data_dir)
print('Input CSV:', input_path.name)
print('Python:', sys.version.split()[0])
print('Execution mode:', 'Colab remote runtime' if is_colab_runtime else 'local Conda/Jupyter')

## CRISP-DM 1 — Business understanding

**Decision context.** The analysis supports decisions about whether the webinar's delivery, materials, technical setup, and learning relevance should be retained or improved in future sessions. The audience is the manuscript team and webinar organizers.

**Primary objective.** Describe post-event participant perceptions using the 13-item Likert matrix, two separate summary ratings, and analyst-coded open-ended feedback.

**Decision questions.** (1) Which experience dimensions were strongest and weakest? (2) How stable are the summaries when the two scale-direction anomalies are normalized? (3) What did participants report learning or valuing? (4) What concrete improvements are supported by both rating patterns and written feedback?

**Success criteria.** Every reported metric has an explicit numerator, denominator, scale rule, and uncertainty or limitation note; every transformation is reproducible; qualitative codes remain separate from quantitative ratings; and no result is interpreted as objective learning, causality, or population-level generalizability.

## CRISP-DM 2 — Data understanding

The source is a desensitized Google Forms CSV. Its grain is one respondent submission. The analysis uses the 13 experience statements as the primary Likert block, keeps overall rating and satisfaction as separate single-item measures, and treats the four open-ended prompts as a qualitative evidence stream. The profile below checks shape, schema, missingness, range validity, identifier uniqueness, and response coverage before interpretation.

In [ ]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import Image, display

from ANALYSIS.src.pipeline.config import (
    PRIMARY_ITEM_KEYS, PRIMARY_ITEM_LABELS, SUMMARY_KEYS, TEXT_KEYS,
)
from ANALYSIS.src.pipeline.io import build_rating_frame, build_text_frame, load_evaluation_csv
from ANALYSIS.src.pipeline.quality import flag_scale_direction_rows, validate_frame

output_dir = PROJECT_ROOT / 'ANALYSIS' / 'outputs'
tables = output_dir / 'tables'
figures = output_dir / 'figures'
for path in (tables, figures, output_dir / 'processed'):
    path.mkdir(parents=True, exist_ok=True)

raw = load_evaluation_csv(input_path)
ratings = build_rating_frame(raw)
text = build_text_frame(raw)
quality = validate_frame(raw, ratings)
timestamp_column = raw.columns[1]
institution_column = raw.columns[5]
text_response_counts = {
    key: int(text[key].ne('').sum()) for key in TEXT_KEYS
}
profile = pd.DataFrame([
    {'check': 'respondent rows', 'value': len(raw), 'definition': 'One row per submitted response'},
    {'check': 'raw columns excluding generated respondent_id', 'value': quality['raw_columns'], 'definition': 'Fields in the supplied CSV'},
    {'check': 'primary Likert items', 'value': len(PRIMARY_ITEM_KEYS), 'definition': 'Normalized 1-5 experience statements'},
    {'check': 'summary rating items', 'value': len(SUMMARY_KEYS), 'definition': 'Overall rating and satisfaction, retained separately'},
    {'check': 'open-ended prompts', 'value': len(TEXT_KEYS), 'definition': 'Qualitative response fields'},
    {'check': 'missing rating cells', 'value': quality['invalid_rating_cells'], 'definition': 'Missing values in the 15 rating fields'},
    {'check': 'out-of-range rating cells', 'value': quality['out_of_range_cells'], 'definition': 'Values outside the 1-5 response range'},
    {'check': 'duplicate generated respondent IDs', 'value': int(raw['respondent_id'].duplicated().sum()), 'definition': 'Uniqueness check for de-identified row IDs'},
    {'check': 'duplicate timestamps', 'value': int(raw[timestamp_column].duplicated().sum()), 'definition': 'Diagnostic only; timestamp is not used as an identifier'},
])
display(profile)
display(pd.DataFrame([
    {'prompt': key, 'nonblank_responses': count, 'response_pct': round(count / len(text) * 100, 2)}
    for key, count in text_response_counts.items()
]))
schema = pd.DataFrame([
    {'stream': 'metadata', 'fields': f'{timestamp_column}; {institution_column}'},
    {'stream': 'quantitative', 'fields': '13 primary Likert items; overall_rating; satisfaction'},
    {'stream': 'qualitative', 'fields': ', '.join(TEXT_KEYS)},
])
display(schema)
profile.to_csv(tables / 'crispdm_data_profile.csv', index=False)
pd.DataFrame([quality]).to_csv(tables / 'crispdm_validation_summary.csv', index=False)

## CRISP-DM 3 — Data preparation

The reusable pipeline reads the raw file without overwriting it, assigns de-identified row IDs, parses the first valid 1–5 response from each rating cell, and extracts open-ended fields under stable names. The scale-direction rule is deliberately narrow: a respondent is flagged only when all 13 primary items are raw 1 while both summary ratings are at least 4. For flagged rows, only the primary matrix is reverse-scored with `6 - raw`; overall rating and satisfaction remain unchanged as anchors. Thematic coding uses the existing codebook and de-identified coding decisions in `ANALYSIS/src/themes/`.

In [ ]:
from ANALYSIS.src.pipeline.run_analysis import run_analysis

manifest = run_analysis(input_path=input_path, output_dir=output_dir)
normalized = pd.read_csv(output_dir / 'processed' / 'normalized_ratings.csv')
flags = pd.read_csv(tables / 'scale_direction_flags.csv')
audit_path = tables / 'normalization_audit.csv'
audit = pd.read_csv(audit_path) if audit_path.stat().st_size else pd.DataFrame()
likert = pd.read_csv(tables / 'likert_item_summary.csv')
summary = pd.read_csv(tables / 'overall_satisfaction_summary.csv')
themes = pd.read_csv(tables / 'thematic_counts.csv')
matrix = pd.read_csv(tables / 'mixed_methods_evidence_matrix.csv')

print('Rows:', manifest['rows'])
print('Flagged scale-direction rows:', manifest['flagged_rows'])
print('Normalization scope:', manifest['normalization_scope'])
display(flags.loc[flags['flagged']].copy())
display(audit.head(10))

## CRISP-DM 4 — Modeling and analysis

Because this is a one-time, nonexperimental post-event evaluation, modeling is defined here as descriptive and exploratory evidence generation rather than predictive machine learning. The notebook reports distributions, means, medians, dispersion, agreement and top-box rates, bootstrap intervals for means, Wilson intervals for proportions, a descriptive 13-item composite, item rankings, institutional reach, and a speaker-clarity-by-overall-rating cross-tab. These are associative summaries, not causal estimates.

In [ ]:
RANDOM_SEED = 20261012
BOOTSTRAP_REPS = 2000
rng = np.random.default_rng(RANDOM_SEED)

def wilson_interval(successes, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = successes / n
    denominator = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denominator
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denominator
    return (100 * (centre - margin), 100 * (centre + margin))

def bootstrap_mean_interval(values, generator, reps=BOOTSTRAP_REPS):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return (np.nan, np.nan)
    draws = generator.choice(values, size=(reps, values.size), replace=True)
    means = draws.mean(axis=1)
    return tuple(np.quantile(means, [0.025, 0.975]))

def metric_row(frame, key, label, stream):
    values = frame[key].dropna().astype(float).to_numpy()
    n = values.size
    agree_n = int((values >= 4).sum())
    top_box_n = int((values == 5).sum())
    agree_low, agree_high = wilson_interval(agree_n, n)
    mean_low, mean_high = bootstrap_mean_interval(values, rng)
    return {
        'stream': stream, 'item_key': key, 'item_label': label, 'n': int(n),
        'mean': round(float(values.mean()), 4), 'median': float(np.median(values)),
        'sd': round(float(values.std(ddof=1)), 4) if n > 1 else np.nan,
        'min': int(values.min()), 'max': int(values.max()),
        'agree_n_4_or_5': agree_n, 'agree_pct_4_or_5': round(100 * agree_n / n, 2),
        'agree_ci95_low': round(agree_low, 2), 'agree_ci95_high': round(agree_high, 2),
        'top_box_n_5': top_box_n, 'top_box_pct_5': round(100 * top_box_n / n, 2),
        'mean_bootstrap_ci95_low': round(float(mean_low), 4),
        'mean_bootstrap_ci95_high': round(float(mean_high), 4),
    }

metric_rows = [
    metric_row(normalized, key, label, 'primary_likert')
    for key, label in zip(PRIMARY_ITEM_KEYS, PRIMARY_ITEM_LABELS)
] + [
    metric_row(normalized, key, key.replace('_', ' ').title(), 'summary_rating')
    for key in SUMMARY_KEYS
]
quantitative_metrics = pd.DataFrame(metric_rows)
display(quantitative_metrics)
quantitative_metrics.to_csv(tables / 'crispdm_quantitative_metrics.csv', index=False)

In [ ]:
analysis_frame = normalized.copy()
analysis_frame['primary_likert_mean'] = analysis_frame[PRIMARY_ITEM_KEYS].mean(axis=1)
analysis_frame['primary_likert_median'] = analysis_frame[PRIMARY_ITEM_KEYS].median(axis=1)
composite_summary = pd.DataFrame([{
    'metric': '13-item normalized experience composite',
    'n': int(analysis_frame['primary_likert_mean'].notna().sum()),
    'mean': round(float(analysis_frame['primary_likert_mean'].mean()), 4),
    'median': round(float(analysis_frame['primary_likert_mean'].median()), 4),
    'sd': round(float(analysis_frame['primary_likert_mean'].std(ddof=1)), 4),
    'min': round(float(analysis_frame['primary_likert_mean'].min()), 4),
    'max': round(float(analysis_frame['primary_likert_mean'].max()), 4),
    'agree_composite_pct': round(float((analysis_frame['primary_likert_mean'] >= 4).mean() * 100), 2),
}])
display(composite_summary)

institution_values = raw[institution_column].fillna('').astype(str).str.strip().replace('', 'Unspecified/NA')
institution_summary = institution_values.value_counts(dropna=False).rename_axis('institution').reset_index(name='n')
institution_summary['pct'] = (institution_summary['n'] / len(raw) * 100).round(2)
display(institution_summary)

clarity_key = 'item_04'
clarity_labels = {1: '1 - Strongly disagree', 2: '2', 3: '3', 4: '4', 5: '5 - Strongly agree'}
speaker_cross = pd.crosstab(normalized[clarity_key], normalized['overall_rating'])
speaker_cross = speaker_cross.reindex(index=range(1, 6), columns=range(1, 6), fill_value=0)
speaker_cross.index = [clarity_labels[i] for i in speaker_cross.index]
display(speaker_cross)
speaker_summary = (
    normalized.groupby(clarity_key, dropna=False)['overall_rating']
    .agg(n='count', mean_overall='mean', top_box_n=lambda s: int((s == 5).sum()), top_box_pct=lambda s: float((s == 5).mean() * 100))
    .reset_index()
)
speaker_summary['clarity_label'] = speaker_summary[clarity_key].map(clarity_labels)
speaker_summary['top_box_pct'] = speaker_summary['top_box_pct'].round(2)
display(speaker_summary)
speaker_spearman = normalized[[clarity_key, 'overall_rating']].corr(method='spearman').iloc[0, 1]
association_summary = pd.DataFrame([{'comparison': 'speaker clarity vs overall rating', 'n': int(normalized[[clarity_key, 'overall_rating']].dropna().shape[0]), 'spearman_rho': round(float(speaker_spearman), 4), 'interpretation': 'Descriptive ordinal association; not causal.'}])
display(association_summary)

composite_summary.to_csv(tables / 'crispdm_composite_summary.csv', index=False)
institution_summary.to_csv(tables / 'crispdm_institution_summary.csv', index=False)
speaker_cross.to_csv(tables / 'crispdm_speaker_clarity_overall_crosstab.csv')
speaker_summary.to_csv(tables / 'crispdm_speaker_clarity_overall_summary.csv', index=False)
association_summary.to_csv(tables / 'crispdm_association_summary.csv', index=False)

## CRISP-DM 5 — Evaluation and validation

The evaluation stage checks that the analysis is internally consistent and that the normalization did not alter the two summary anchors. It also quantifies sensitivity by comparing raw and normalized primary-item means. These checks validate computation; they do not validate the original respondents' intent or establish psychometric validity.

In [ ]:
raw_item_means = ratings[PRIMARY_ITEM_KEYS].astype(float).mean()
normalized_item_means = normalized[PRIMARY_ITEM_KEYS].astype(float).mean()
normalization_sensitivity = pd.DataFrame({
    'item_key': PRIMARY_ITEM_KEYS,
    'raw_mean': [raw_item_means[key] for key in PRIMARY_ITEM_KEYS],
    'normalized_mean': [normalized_item_means[key] for key in PRIMARY_ITEM_KEYS],
})
normalization_sensitivity['mean_change'] = normalization_sensitivity['normalized_mean'] - normalization_sensitivity['raw_mean']
normalization_sensitivity = normalization_sensitivity.round(4)
display(normalization_sensitivity)
normalization_sensitivity.to_csv(tables / 'crispdm_normalization_sensitivity.csv', index=False)

checks = pd.DataFrame([
    {'check': 'raw and normalized row counts match', 'passed': len(raw) == len(normalized)},
    {'check': 'all rating cells remain in 1-5 range', 'passed': bool(((normalized[PRIMARY_ITEM_KEYS + SUMMARY_KEYS] >= 1) & (normalized[PRIMARY_ITEM_KEYS + SUMMARY_KEYS] <= 5)).all().all())},
    {'check': 'summary ratings unchanged by normalization', 'passed': np.array_equal(ratings[SUMMARY_KEYS].to_numpy(), normalized[SUMMARY_KEYS].to_numpy())},
    {'check': 'manifest row count matches source', 'passed': int(manifest['rows']) == len(raw)},
    {'check': 'flag count matches manifest', 'passed': int(flags['flagged'].sum()) == int(manifest['flagged_rows'])},
    {'check': 'audit rows equal flagged rows times 13', 'passed': len(audit) == int(manifest['flagged_rows']) * len(PRIMARY_ITEM_KEYS)},
])
display(checks)
if not bool(checks['passed'].all()):
    raise AssertionError('At least one reproducibility or normalization check failed.')

## Mixed-methods integration: qualitative evidence

The thematic stream is analyzed at the unit of one respondent answer to one open-ended prompt. Multiple codes are allowed for distinct ideas in one answer. Counts therefore represent coded assignments and respondent coverage, not mutually exclusive participant categories. The codebook and coding decisions are retained as auditable project assets.

In [ ]:
theme_decisions = pd.read_csv(PROJECT_ROOT / 'ANALYSIS' / 'src' / 'themes' / 'coding_decisions.csv')
theme_counts = themes.copy()
theme_counts['respondent_coverage_pct'] = (theme_counts['n_respondents'] / len(raw) * 100).round(2)
theme_counts['assignment_share_pct'] = (theme_counts['n_assignments'] / theme_counts['n_assignments'].sum() * 100).round(2)
display(theme_counts)
theme_by_prompt = pd.crosstab(theme_decisions['source_field'], theme_decisions['code_id'])
display(theme_by_prompt)
theme_counts.to_csv(tables / 'crispdm_theme_counts_with_denominators.csv', index=False)
theme_by_prompt.to_csv(tables / 'crispdm_theme_by_prompt.csv')

In [ ]:
item_plot = quantitative_metrics[quantitative_metrics['stream'] == 'primary_likert'].sort_values('mean')
fig, ax = plt.subplots(figsize=(10, 7))
ax.errorbar(
    item_plot['mean'], item_plot['item_label'],
    xerr=[item_plot['mean'] - item_plot['mean_bootstrap_ci95_low'], item_plot['mean_bootstrap_ci95_high'] - item_plot['mean']],
    fmt='o', color='#2f6690', ecolor='#7a9db8', capsize=3,
)
ax.set_xlim(1, 5)
ax.set_xlabel('Mean normalized rating (1-5), with bootstrap 95% interval')
ax.set_ylabel('')
ax.set_title('Primary Likert item performance')
fig.tight_layout()
fig.savefig(figures / 'crispdm_item_means_with_ci.png', dpi=180)
plt.show()

plt.figure(figsize=(7, 5))
sns.heatmap(speaker_cross, annot=True, fmt='g', cmap='Blues')
plt.xlabel('Overall rating')
plt.ylabel('Speaker clarity rating')
plt.title('Speaker clarity by overall rating: respondent counts')
plt.tight_layout()
plt.savefig(figures / 'crispdm_speaker_clarity_overall_heatmap.png', dpi=180)
plt.show()

## CRISP-DM 6 — Deployment and decision handoff

Deployment here means producing traceable tables, figures, a manifest, and a decision-oriented readout for Chapter 3 and Chapter 4. The generated prompts below are evidence pointers for human interpretation; they are not automatically written conclusions. Re-run the notebook after any approved data, coding, or pipeline change.

In [ ]:
top_item = item_plot.iloc[-1]
lowest_item = item_plot.iloc[0]
substantive_theme_counts = theme_counts.loc[theme_counts['code_id'].ne('NO_SUBSTANTIVE_RESPONSE')].copy()
top_theme = substantive_theme_counts.sort_values(['n_assignments', 'code_id'], ascending=[False, True]).iloc[0] if len(substantive_theme_counts) else None
decision_readout = {
    'sample_n': int(len(raw)),
    'flagged_scale_direction_rows': int(manifest['flagged_rows']),
    'strongest_primary_item': {'item_key': top_item['item_key'], 'label': top_item['item_label'], 'mean': float(top_item['mean']), 'agree_pct': float(top_item['agree_pct_4_or_5'])},
    'lowest_primary_item': {'item_key': lowest_item['item_key'], 'label': lowest_item['item_label'], 'mean': float(lowest_item['mean']), 'agree_pct': float(lowest_item['agree_pct_4_or_5'])},
    'qualitative_priority_code': None if top_theme is None else {'code_id': top_theme['code_id'], 'label': top_theme['label'], 'assignments': int(top_theme['n_assignments']), 'respondents': int(top_theme['n_respondents'])},
    'interpretation_boundary': 'Descriptive post-event perceptions and coded reported feedback; not objective learning, causal impact, or population estimates.',
}
(tables / 'crispdm_decision_readout.json').write_text(json.dumps(decision_readout, indent=2), encoding='utf-8')
decision_prompts = pd.DataFrame([
    {'decision_area': 'Retain or showcase strength', 'evidence': f"{top_item['item_label']} (mean {top_item['mean']:.2f}; agreement {top_item['agree_pct_4_or_5']:.2f}%)"},
    {'decision_area': 'Investigate improvement opportunity', 'evidence': f"{lowest_item['item_label']} (mean {lowest_item['mean']:.2f}; agreement {lowest_item['agree_pct_4_or_5']:.2f}%)"},
    {'decision_area': 'Triangulate with participant voice', 'evidence': 'Compare the lowest-rated dimensions with thematic codes and prompt-specific response coverage before recommending changes.'},
    {'decision_area': 'Protect data quality', 'evidence': f"Review the {int(manifest['flagged_rows'])} scale-direction flag(s) and preserve the normalization audit with every manuscript result."},
])
display(decision_prompts)
decision_prompts.to_csv(tables / 'crispdm_decision_prompts.csv', index=False)
print('CRISP-DM outputs written to:', output_dir)

## Copy-ready quantitative and mixed-methods findings

This cell creates a compact reporting layer using the verified tables. It is intended as a drafting aid for Results and Discussion, not as a substitute for checking the denominators, intervals, assumptions, and qualitative evidence matrix.

In [ ]:
def summary_row(key):
    return summary.loc[summary['item_key'].eq(key)].iloc[0]

overall_row = summary_row('overall_rating')
satisfaction_row = summary_row('satisfaction')
strong_clarity = speaker_summary.loc[speaker_summary[clarity_key].eq(5)]
strong_clarity_text = 'not estimable' if strong_clarity.empty else f"{float(strong_clarity.iloc[0]['top_box_pct']):.2f}% ({int(strong_clarity.iloc[0]['top_box_n'])}/{int(strong_clarity.iloc[0]['n'])})"

report_lines = [
    '### Key performance highlights',
    f"- The analytic sample contained {len(raw)} submitted responses. The 13-item Likert matrix was summarized after the pre-specified normalization rule flagged {int(manifest['flagged_rows'])} respondent row(s).",
    f"- Overall satisfaction: {int(satisfaction_row['count_5'])}/{int(satisfaction_row['n'])} respondents ({float(satisfaction_row['pct_5']):.2f}%) selected 5/5; {float(satisfaction_row['agree_pct']):.2f}% selected 4 or 5.",
    f"- Overall session rating: {int(overall_row['count_5'])}/{int(overall_row['n'])} respondents ({float(overall_row['pct_5']):.2f}%) selected 5/5; {float(overall_row['agree_pct']):.2f}% selected 4 or 5.",
    f"- Highest primary dimension: {top_item['item_label']} (mean {float(top_item['mean']):.2f}; agreement {float(top_item['agree_pct_4_or_5']):.2f}%; top-box {float(top_item['top_box_pct_5']):.2f}%).",
    f"- Lowest primary dimension: {lowest_item['item_label']} (mean {float(lowest_item['mean']):.2f}; agreement {float(lowest_item['agree_pct_4_or_5']):.2f}%; top-box {float(lowest_item['top_box_pct_5']):.2f}%).",
    f"- Among respondents selecting the strongest speaker-clarity category, the overall-rating 5/5 rate was {strong_clarity_text}; this is a descriptive cross-tab, not evidence of a causal effect.",
    f"- The ordinal association between speaker clarity and overall rating was Spearman rho = {float(association_summary.iloc[0]['spearman_rho']):.4f} across {int(association_summary.iloc[0]['n'])} complete cases; interpret this as descriptive, not causal.",
    '',
    '### Qualitative evidence and action cues',
    '- Theme frequencies are coded assignments and may include multiple codes per respondent answer; they are not counts of objectively learned outcomes.',
    f"- The institutional distribution and prompt-level response coverage are available in `crispdm_institution_summary.csv` and `crispdm_data_profile.csv`.",
    f"- The mixed-methods evidence matrix is available in `mixed_methods_evidence_matrix.csv`; use it to compare rating strengths and weaknesses with reported learning, valued features, and improvement requests.",
    '',
    '### Interpretation boundary',
    '- These results describe post-event perceptions from this respondent set. They do not establish objective learning, causal impact, psychometric validity, or population-level generalizability.',
]
report_text = '\n'.join(report_lines)
print(report_text)
(tables / 'crispdm_copy_ready_findings.md').write_text(report_text + '\n', encoding='utf-8')

In [ ]:
import pandas as pd

tables = output_dir / 'tables'
likert = pd.read_csv(tables / 'likert_item_summary.csv')
summary = pd.read_csv(tables / 'overall_satisfaction_summary.csv')
themes = pd.read_csv(tables / 'thematic_counts.csv')
display(likert[['item_order', 'item_label', 'n', 'mean', 'sd', 'agree_pct']])
display(summary[['item_key', 'n', 'mean', 'sd', 'agree_pct']])
display(themes)

In [ ]:
from IPython.display import Image, display

figures = output_dir / 'figures'
display(Image(filename=str(figures / 'likert_item_means.png')))
display(Image(filename=str(figures / 'likert_distribution.png')))

In [ ]:
matrix = pd.read_csv(tables / 'mixed_methods_evidence_matrix.csv')
display(matrix)
print('Copy-ready quantitative table:')
print(likert[['item_order', 'item_label', 'n', 'mean', 'sd', 'agree_pct']].to_markdown(index=False))
print('Copy-ready summary table:')
print(summary[['item_key', 'n', 'mean', 'sd', 'agree_pct']].to_markdown(index=False))
print('Copy-ready thematic counts:')
print(themes.to_markdown(index=False))

## Interpretation boundary

The normalized Likert values describe respondents' post-event perceptions. Overall rating and satisfaction are retained as separate single-item summaries. The two flagged rows are corrected only for the 13 primary Likert items because their all-1 item pattern conflicts with their positive summary ratings. Open-ended code frequencies describe reported feedback and are not objective measures of learning. The design is nonexperimental and post-event, so these outputs do not establish causation, objective competence, or population-level generalizability.